In [29]:
from xaikd import models, datasets, utils

from torchmetrics.classification import BinaryAUROC
from tqdm.autonotebook import tqdm


import numpy as np

In [13]:
DEVICE = utils.get_device()

In [14]:
dataset = datasets.construct("celeba")
dl_val = datasets.build_dataloader(dataset.create_subset(train_split=False), shuffle=False)

/home/pat/projects/xai-kd/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:561: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [30]:
def eval_celeba(model_name):
    model = models.get_trained_model(model_name).eval().to(DEVICE)
    
    arr_metrics = [] 
    for tix in range(datasets.celeba.NUM_CELEBA_ATTRIBUTES):
        arr_metrics.append(BinaryAUROC(thresholds=20))

    for x, y in tqdm(dl_val, desc=f"model={model_name}"):
        
        arr_logits = model(x.to(DEVICE))
        
        for tix in range(datasets.celeba.NUM_CELEBA_ATTRIBUTES):
            y_task = y[:, tix]
            logits_task = arr_logits[:, tix]
            
            arr_metrics[tix].update(
                logits_task.detach().cpu(), 
                y_task.detach().long().cpu()
            )
    
    print(f"Evaluating model={model_name}")
    for tix in range(datasets.celeba.NUM_CELEBA_ATTRIBUTES):
        auroc = arr_metrics[tix].compute()
        auroc = np.max([auroc, 1-auroc])
        arr_metric[tix] = auroc
        print(f"> auroc(val attr{tix})={auroc:.4f}")
    print(f"avg(val auroc)={np.mean(arr_metrics):.4f}")

In [28]:
for name in [
    "celeba-resnet18-finetunedv1",
    "celeba-resnet50-finetunedv1",
    "celeba-wideresnet50_2-finetunedv1",
    "celeba-vitb16-finetunedv1",
]:
    eval_celeba(name)

model=celeba-resnet18-finetunedv1: 100%|█████████████████████████████████████████████████| 311/311 [06:58<00:00,  1.35s/it]


Evaluating model=ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU

NameError: name 'np' is not defined